# Hamiltonian simulation

Trotterize a transverse-field Ising chain and compare an observable trajectory.

## What you will learn

- How to express this workflow with Qiskit's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    print_scaling_table,
    qiskit_selection,
    total_variation_distance,
)

## 1. Define the quantum problem

Product-formula time evolution approximates a Hamiltonian with a sequence of local exponentials.

In [2]:
times = np.linspace(0.0, 1.2, 13)
observable = SparsePauliOp("IIZ")

def evolution_circuit(time_value):
    circuit = QuantumCircuit(3)
    circuit.x(0)
    steps = 6
    dt = float(time_value) / steps
    for _ in range(steps):
        circuit.rzz(1.1 * dt, 0, 1)
        circuit.rzz(1.1 * dt, 1, 2)
        for wire in range(3):
            circuit.rx(0.7 * dt, wire)
    return circuit

circuits = [evolution_circuit(value) for value in times]

def reference_trajectory():
    estimator = StatevectorEstimator()
    return np.asarray([estimator.run([(c, observable)]).result()[0].data.evs.item() for c in circuits])

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_trajectory)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def mettleq_trajectory():
    return np.asarray([estimator.run([(c, observable)]).result()[0].data.evs.item() for c in compiled])

candidate, mettleq_ms, _ = benchmark(mettleq_trajectory)
error = max_abs_error(reference, candidate)
method, device = qiskit_selection(estimator)

## 4. Check correctness before discussing speed

The observable trajectory over all requested times must agree, exposing errors that a single endpoint could hide.

In [5]:
tutorial_result = emit_result(
    notebook="qiskit/12_hamiltonian_simulation.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="observable trajectory atol=3e-6",
    passed=error <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_observable_error": error, "times": times, "reference": reference, "mettleq": candidate},
)


Comparison summary
------------------
Correctness contract: PASS — observable trajectory atol=3e-6
SDK reference median: 9.321 ms
MettleQ median:       46.800 ms
Timing interpretation: the SDK reference was 5.021x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "observable trajectory atol=3e-6", "exact_match": false, "framework": "qiskit", "machine": "arm64", "metrics": {"max_observable_error": 1.2218022292787012e-06, "mettleq": [-1.0, -0.997553825378418, -0.9902541637420654, -0.9782247543334961, -0.9616613984107971, -0.9408406019210815, -0.9161059856414795, -0.8878596425056458, -0.8565607666969299, -0.8227137923240662, -0.7868613600730896, -0.7495623230934143, -0.71140056848526], "reference": [-1.0, -0.9975533999646712, -0.9902542917948219, -0.9782239468098275, -0.9616618196615209, -0.9408416125730155, -0.916105884970

## What should you conclude?

Deeper, wider dynamics are a natural GPU workload once state evolution dominates construction and readback.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.